## DuckLake First Look

💡 Instructions : Start the container with **Docker Compose** and connect to the Jupyter Server at `localhost:8888/lab`

In [1]:
import duckdb
import sqlite3
from pathlib import Path

### Creating a DuckLake

Create a new DuckDB Database

In [2]:
db = duckdb.connect('br_electorate.duckdb')

Install the `DuckLake` and `SQLite` extensions

In [3]:
db.sql('''
    INSTALL ducklake;
    INSTALL sqlite;
''')

Create the DuckLake database with SQLite as the Metadata Store

In [4]:
db.sql('''
    ATTACH 'ducklake:sqlite:/data/electorate/metadata.sqlite' AS electorate (DATA_PATH '/data/electorate/');
''')

In [5]:
db.sql('''
    SHOW databases;
''')

┌────────────────────────────────┐
│         database_name          │
│            varchar             │
├────────────────────────────────┤
│ __ducklake_metadata_electorate │
│ br_electorate                  │
│ electorate                     │
└────────────────────────────────┘

Use the newly created database

In [6]:
db.sql('USE electorate;')

### 01 - Create New Table _electorate_history_

Create table `electorate_history` with the records from the electorate in 2012.

In [7]:
db.sql(
'''
    CREATE OR REPLACE TABLE electorate_history AS (
        SELECT
            ANO_ELEICAO,
            SG_UF,
            NR_ZONA,
            NR_SECAO,
            QT_ELEITORES_PERFIL
        FROM read_csv('/data/2012/extracted/*/*.csv', delim=';', encoding='latin-1')
    )
'''
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Inspect the Metadata**

In [8]:
db.sql(
'''
    SELECT
    *
    FROM __ducklake_metadata_electorate.main.ducklake_table
'''
)

┌──────────┬──────────────────────────────────────┬────────────────┬──────────────┬───────────┬────────────────────┬─────────────────────┬──────────────────┐
│ table_id │              table_uuid              │ begin_snapshot │ end_snapshot │ schema_id │     table_name     │        path         │ path_is_relative │
│  int64   │               varchar                │     int64      │    int64     │   int64   │      varchar       │       varchar       │      int64       │
├──────────┼──────────────────────────────────────┼────────────────┼──────────────┼───────────┼────────────────────┼─────────────────────┼──────────────────┤
│        1 │ 019d187f-cc49-7c58-937b-0eb86b503620 │              1 │         NULL │         0 │ electorate_history │ electorate_history/ │                1 │
└──────────┴──────────────────────────────────────┴────────────────┴──────────────┴───────────┴────────────────────┴─────────────────────┴──────────────────┘

**Inspect the Parquet Files created**

In [9]:
!tree /data/electorate

/data/electorate
├── main
│   └── electorate_history
│       └── ducklake-019d187f-ccec-7741-a831-913cb1a85abf.parquet
└── metadata.sqlite

3 directories, 2 files


### 02 - Insert new records into the Table

> ⚠️ This execution may take a few minutes

In [11]:
db.sql(
'''
    INSERT INTO electorate_history (
        ANO_ELEICAO,
        SG_UF,
        NR_ZONA,
        NR_SECAO,
        QT_ELEITORES_PERFIL
    )
    SELECT
        ANO_ELEICAO,
        SG_UF,
        NR_ZONA,
        NR_SECAO,
        QT_ELEITORES_PERFIL
    FROM read_csv(
        [
            '/data/2014/extracted/*/*.csv',
            '/data/2016/extracted/*/*.csv',
            '/data/2018/extracted/*/*.csv',
            '/data/2020/extracted/*/*.csv',
            '/data/2022/extracted/*/*.csv',
            '/data/2024/extracted/*/*.csv',
            '/data/ATUAL/extracted/*/*.csv'
        ],
        delim=';',
        encoding='latin-1', 
        union_by_name = true
    )
'''
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**Show and Count the records inserted**

In [12]:
db.sql(
'''
    SELECT 
        *
    FROM electorate_history
    WHERE ANO_ELEICAO > 2012
    LIMIT 10
'''
)

┌─────────────┬─────────┬─────────┬──────────┬─────────────────────┐
│ ANO_ELEICAO │  SG_UF  │ NR_ZONA │ NR_SECAO │ QT_ELEITORES_PERFIL │
│    int64    │ varchar │  int64  │  int64   │        int64        │
├─────────────┼─────────┼─────────┼──────────┼─────────────────────┤
│        2014 │ AL      │      31 │       91 │                   2 │
│        2014 │ AL      │      31 │       91 │                   3 │
│        2014 │ AL      │      31 │       91 │                   1 │
│        2014 │ AL      │      31 │       91 │                   1 │
│        2014 │ AL      │      31 │       91 │                   1 │
│        2014 │ AL      │      31 │       91 │                   1 │
│        2014 │ AL      │      31 │       91 │                   1 │
│        2014 │ AL      │      31 │       91 │                   1 │
│        2014 │ AL      │      31 │       91 │                   1 │
│        2014 │ AL      │      31 │       91 │                   1 │
└─────────────┴─────────┴─────────

In [13]:
db.sql(
'''
    SELECT 
        ANO_ELEICAO, COUNT(*)
    FROM electorate_history
    GROUP BY ANO_ELEICAO 
    ORDER BY ANO_ELEICAO
'''
)

┌─────────────┬──────────────┐
│ ANO_ELEICAO │ count_star() │
│    int64    │    int64     │
├─────────────┼──────────────┤
│        2012 │     54953845 │
│        2014 │     60848095 │
│        2016 │     64969118 │
│        2018 │     68819771 │
│        2020 │     72869730 │
│        2022 │     74897799 │
│        2024 │     82360461 │
│        9999 │     85811628 │
└─────────────┴──────────────┘

**Inspect the Metadata** - Snapshots

In [15]:
db.sql(
'''
    SELECT
    *
    FROM __ducklake_metadata_electorate.main.ducklake_snapshot
'''
)

┌─────────────┬───────────────────────────────┬────────────────┬─────────────────┬──────────────┐
│ snapshot_id │         snapshot_time         │ schema_version │ next_catalog_id │ next_file_id │
│    int64    │            varchar            │     int64      │      int64      │    int64     │
├─────────────┼───────────────────────────────┼────────────────┼─────────────────┼──────────────┤
│           0 │ 2026-03-23 02:21:57.346491+00 │              0 │               1 │            0 │
│           1 │ 2026-03-23 02:22:01.682978+00 │              1 │               2 │            1 │
│           2 │ 2026-03-23 02:28:25.549585+00 │              1 │               2 │            3 │
└─────────────┴───────────────────────────────┴────────────────┴─────────────────┴──────────────┘

What each snapshot did?

In [17]:
db.sql(
'''
    SELECT
    *
    FROM __ducklake_metadata_electorate.main.ducklake_snapshot_changes
'''
)

┌─────────────┬─────────────────────────────────────────────────────────────────┬─────────┬────────────────┬───────────────────┐
│ snapshot_id │                          changes_made                           │ author  │ commit_message │ commit_extra_info │
│    int64    │                             varchar                             │ varchar │    varchar     │      varchar      │
├─────────────┼─────────────────────────────────────────────────────────────────┼─────────┼────────────────┼───────────────────┤
│           0 │ created_schema:"main"                                           │ NULL    │ NULL           │ NULL              │
│           1 │ created_table:"main"."electorate_history",inserted_into_table:1 │ NULL    │ NULL           │ NULL              │
│           2 │ inserted_into_table:1                                           │ NULL    │ NULL           │ NULL              │
└─────────────┴─────────────────────────────────────────────────────────────────┴─────────┴──────

**Inspect the Parquet Files created**

In [16]:
!tree /data/electorate

/data/electorate
├── main
│   └── electorate_history
│       ├── ducklake-019d187f-ccec-7741-a831-913cb1a85abf.parquet
│       ├── ducklake-019d1886-3f09-7946-902e-207ddf28bc05.parquet
│       └── ducklake-019d188d-a4d5-7a6f-852d-890cf3589534.parquet
└── metadata.sqlite

3 directories, 4 files
